# 🏛️ LexiMini AI — Knowledge Distillation
**Teacher:** Fine-tuned Gemma 4 E4B (Indian Legal)
**Student:** Gemma 3 1B
**Method:** Tunix Logit-Based Distillation
**Platform:** Kaggle TPU v5e-8

> Transfer legal knowledge from 4B teacher to 1B student using Google Tunix

## CELL 1 — Install Libraries

In [ ]:
!pip install -q kagglehub
!pip install -q tensorflow tensorboardX
!pip install -q grain-nightly datasets
!pip install -q git+https://github.com/google/tunix
!pip install -q git+https://github.com/google/qwix
!pip uninstall -q -y flax
!pip install -q git+https://github.com/google/flax.git
!pip install -q google-cloud-storage sentencepiece
print(' Done')

## CELL 2 — Imports + Config

In [ ]:
import gc, os, json
import numpy as np
import jax
import jax.numpy as jnp
import optax
import kagglehub
from flax import nnx
from orbax import checkpoint as ocp
from tunix.distillation import distillation_trainer
from tunix.distillation import strategies
from tunix.generate import sampler as sampler_lib
from tunix.generate import tokenizer_adapter as tokenizer_lib
from tunix.models.gemma3 import model as gemma3_lib
from tunix.models.gemma3 import params_safetensors as params_lib
from tunix.sft import utils

# ── Config ────────────────────────────────────────────────
BUCKET_NAME      = 'leximini-training-data'
HF_TOKEN         = 'your-hf-token'   
KAGGLE_USERNAME  = 'your-kaggle-username'  

# Paths
TEACHER_CKPT_DIR = '/kaggle/working/teacher_ckpt/'
STUDENT_CKPT_DIR = '/kaggle/working/student_ckpt/'
OUTPUT_DIR       = '/kaggle/working/leximini-distilled/'

# Distillation hyperparams
BATCH_SIZE       = 4
MAX_LENGTH       = 256
MAX_STEPS        = 500
EVAL_EVERY       = 100
LEARNING_RATE    = 1e-4
TEMPERATURE      = 2.0   
ALPHA            = 0.7   

# TPU Mesh — Kaggle TPU v5e-8
MESH = [(1, 8), ('fsdp', 'tp')]

os.makedirs(TEACHER_CKPT_DIR, exist_ok=True)
os.makedirs(STUDENT_CKPT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'JAX version  : {jax.__version__}')
print(f'TPU devices  : {jax.devices()}')
print(f'Device count : {len(jax.devices())}')

## CELL 3 — From GCS Training Data Download

In [ ]:


TRAIN_PATH = '/kaggle/input/leximini-data/train.jsonl'
EVAL_PATH  = '/kaggle/input/leximini-data/eval.jsonl'

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

train_data = load_jsonl(TRAIN_PATH)
eval_data  = load_jsonl(EVAL_PATH)
print(f' Train: {len(train_data)} | Eval: {len(eval_data)}')

## CELL 4 — Teacher Model Load (Fine-tuned Gemma 4 E4B from HF)

In [ ]:
from huggingface_hub import snapshot_download
import subprocess

os.environ['HF_TOKEN'] = HF_TOKEN

# Fine-tuned teacher model  download from GCS
print('Downloading teacher model (fine-tuned Gemma 4 E4B)...')
# First local copy from GCS
subprocess.run([
    'gsutil', '-m', 'cp', '-r',
    f'gs://{BUCKET_NAME}/checkpoints/leximini-checkpoints/checkpoint-429/*',
    TEACHER_CKPT_DIR
])
print(f' Teacher checkpoint downloaded: {os.listdir(TEACHER_CKPT_DIR)}')

## CELL 5 — Student Model Download (Gemma 3 1B)

In [ ]:
# From Kaggle  Gemma 3 1B download
if 'KAGGLE_USERNAME' not in os.environ:
    kagglehub.login()

print('Downloading student model (Gemma 3 1B)...')
student_path = kagglehub.model_download('google/gemma-3/flax/1b-it')
print(f' Student model downloaded: {student_path}')

## CELL 6 — Load Tokenizer

In [ ]:
import sentencepiece as spm

tokenizer_path = os.path.join(student_path, 'tokenizer.model')
sp = spm.SentencePieceProcessor()
sp.Load(tokenizer_path)

PAD_ID = 0
print(f' Tokenizer loaded | Vocab: {sp.GetPieceSize()}')

## CELL 7 — Legal Dataset Prepare karo

In [ ]:
def tokenize_batch(data, batch_size, max_len):
    batches = []
    batch_tokens, batch_masks = [], []

    for item in data:
        text = item['text']
        split_tag = '<start_of_turn>model\n'
        split_idx = text.find(split_tag)
        prompt = text[:split_idx + len(split_tag)] if split_idx != -1 else text

        full_ids   = sp.Encode(text)[:max_len]
        prompt_len = len(sp.Encode(prompt))

        tokens = np.full((max_len,), PAD_ID, dtype=np.int32)
        tokens[:len(full_ids)] = full_ids

        mask = np.zeros((max_len,), dtype=np.int32)
        end  = min(len(full_ids), max_len)
        if end > prompt_len:
            mask[prompt_len:end] = 1

        batch_tokens.append(tokens)
        batch_masks.append(mask)

        if len(batch_tokens) == batch_size:
            batches.append(distillation_trainer.TrainingInput(
                input_tokens=jnp.array(np.stack(batch_tokens)),
                input_mask  =jnp.array(np.stack(batch_masks))
            ))
            batch_tokens, batch_masks = [], []

    return batches

print('Preparing batches...')
train_batches = tokenize_batch(train_data, BATCH_SIZE, MAX_LENGTH)
eval_batches  = tokenize_batch(eval_data,  BATCH_SIZE, MAX_LENGTH)
print(f' Train: {len(train_batches)} | Eval: {len(eval_batches)}')

## CELL 8 — Teacher + Student Model Load on TPU

In [ ]:
mesh = jax.make_mesh(*MESH, axis_types=(jax.sharding.AxisType.Auto,) * len(MESH[0]))

def get_sharded_model(ckpt_path, model_config, mesh):
    abs_model = nnx.eval_shape(
        lambda: gemma3_lib.Gemma3(model_config, rngs=nnx.Rngs(params=0))
    )
    abs_state = nnx.state(abs_model)
    abs_state = jax.tree.map(
        lambda a, s: jax.ShapeDtypeStruct(a.shape, jnp.bfloat16, sharding=s),
        abs_state,
        nnx.get_named_sharding(abs_state, mesh),
    )
    checkpointer = ocp.StandardCheckpointer()
    restored = checkpointer.restore(ckpt_path, target=abs_state)
    graph_def, _ = nnx.split(abs_model)
    return nnx.merge(graph_def, restored)

# Teacher — Gemma 4 E4B config
print('Loading teacher model...')
teacher_config = gemma3_lib.ModelConfig(
    num_layers=32, num_embed=262208, embed_dim=2560,
    hidden_dim=10240, num_heads=8, num_kv_heads=4,
    head_dim=256, sliding_window_size=1024,
    local_base_frequency=10000, global_base_frequency=1000000,
    local_scale_factor=1.0, global_scale_factor=8.0,
)
with mesh:
    teacher_model = params_lib.create_model_from_safe_tensors(
        TEACHER_CKPT_DIR, teacher_config, mesh
    )
print(' Teacher loaded!')

# Student — Gemma 3 1B
print('Loading student model...')
student_config = gemma3_lib.get_model_config('gemma3_1b')
with mesh:
    student_model = params_lib.create_model_from_safe_tensors(
        os.path.join(student_path, '1b-it'), student_config, mesh
    )
print(' Student loaded!')

## CELL 9 — Distillation Setup

In [ ]:
VOCAB_SIZE = student_config.num_embed

def model_forward_fn(model, input_tokens, input_mask, positions, attention_mask):
    logits, _ = model(input_tokens, positions, None, attention_mask)
    return logits[:, :-1, :]

def labels_fn(input_tokens, input_mask, **kwargs):
    target_tokens = input_tokens[:, 1:]
    target_mask   = input_mask[:, 1:]
    labels = jax.nn.one_hot(target_tokens, VOCAB_SIZE)
    return labels * target_mask.astype(labels.dtype)[..., None]

def gen_model_input_fn(x):
    pad_mask       = x.input_tokens != PAD_ID
    positions      = utils.build_positions_from_mask(pad_mask)
    attention_mask = utils.make_causal_attn_mask(pad_mask)
    return {
        'input_tokens'  : x.input_tokens,
        'input_mask'    : x.input_mask,
        'positions'     : positions,
        'attention_mask': attention_mask,
    }

strategy = strategies.LogitStrategy(
    student_forward_fn = model_forward_fn,
    teacher_forward_fn = model_forward_fn,
    labels_fn          = labels_fn,
    temperature        = TEMPERATURE,
    alpha              = ALPHA,
)

config = distillation_trainer.TrainingConfig(
    eval_every_n_steps        = EVAL_EVERY,
    max_steps                 = MAX_STEPS,
    checkpoint_root_directory = OUTPUT_DIR,
)

optimizer = optax.adamw(LEARNING_RATE)

teacher_model.eval()
student_model.train()

trainer = distillation_trainer.DistillationTrainer(
    student_model  = student_model,
    teacher_model  = teacher_model,
    strategy       = strategy,
    optimizer      = optimizer,
    training_config= config,
).with_gen_model_input_fn(gen_model_input_fn)

print(' Distillation trainer ready!')

## CELL 10 — Distillation Start! 

In [ ]:
print(' Distillation started...')
with mesh:
    trainer.train(
        train_batches * (MAX_STEPS // len(train_batches) + 1),
        eval_batches
    )
print(' Distillation complete!')

## CELL 11 — Student Model Save to GCS

In [ ]:
import subprocess

# Save distilled student model
print('Saving distilled model...')
checkpointer = ocp.StandardCheckpointer()
_, state = nnx.split(student_model)
checkpointer.save(os.path.join(OUTPUT_DIR, 'final'), state)
checkpointer.wait_until_finished()

# GCS pe upload
subprocess.run([
    'gsutil', '-m', 'cp', '-r',
    OUTPUT_DIR,
    f'gs://{BUCKET_NAME}/models/leximini-distilled-1b/'
])
print(f'✅ Distilled model saved to gs://{BUCKET_NAME}/models/leximini-distilled-1b/')

## CELL 12 — Test Distilled Model

In [ ]:
student_model.eval()

sampler = sampler_lib.Sampler(
    transformer  = student_model,
    tokenizer    = tokenizer_lib.Tokenizer(
        tokenizer_type='sentencepiece',
        tokenizer_path=tokenizer_path
    ),
    cache_config = sampler_lib.CacheConfig(
        cache_size  = MAX_LENGTH + 64,
        num_layers  = student_config.num_layers,
        num_kv_heads= student_config.num_kv_heads,
        head_dim    = student_config.head_dim,
    ),
)

def ask_leximini(question):
    prompt = (
        f'<start_of_turn>user\n{question}<end_of_turn>\n'
        f'<start_of_turn>model\n'
    )
    result = sampler([prompt], total_generation_steps=200)
    return result.text[0]

print('=== TEST ===')
print(ask_leximini('What are my rights under BNS Section 173?'))
print()
print(ask_leximini('Agar mujhe bina warrant ke giraftaar kiya jaye toh kya karoon?'))